# 第5章 智能系统部署与实现

> **课程**：智能系统开发教程 —— 第5章  
> **硬件平台**：ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB  
> **软件环境**：CANN 9.0.0, Python 3.11, A2-arm

---

## 课程导览

本章系统介绍智能系统部署与实现的完整技术体系，从模型训练到实际部署的关键环节，涵盖编译器方案、推理框架、CANN 异构计算架构、AscendC 编程、毕昇编译器、运行时与驱动程序等核心组件。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">主题</th>
<th style="text-align: left;">内容概要</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">智能系统部署方案概述</td>
<td style="text-align: left;">训练与推理差异 → 两类技术路线 → 部署流程</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">CANN 异构计算架构</td>
<td style="text-align: left;">架构总览 → 三层逻辑 → 全栈体系 → 开源生态</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">CANN 核心组件</td>
<td style="text-align: left;">算子库 → 通信库HCCL → 图引擎GE → AscendC → 毕昇编译器 → 运行时AscendCL → 驱动</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">模型优化与部署实践</td>
<td style="text-align: left;">量化/剪枝/融合/蒸馏 → CANN应用案例 → 边缘部署</td>
</tr>
</table>

> 本课程穿插可执行代码段，建议在昇腾 NPU 环境下运行体验。

---

## 1. 智能系统部署方案概述

### 1.1 深度学习模型部署的挑战

训练好的深度学习模型要真正产生价值，必须走出训练环境、部署到实际的推理硬件上。然而，训练阶段与推理阶段存在巨大差异：

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">维度</th><th style="text-align: left;">训练阶段</th><th style="text-align: left;">推理阶段</th></tr>
<tr><td style="text-align: left;">环境</td><td style="text-align: left;">高性能服务器/云端</td><td style="text-align: left;">手机/摄像头/车辆/边缘设备</td></tr>
<tr><td style="text-align: left;">关注点</td><td style="text-align: left;">模型精度</td><td style="text-align: left;">计算效率、延迟、功耗、资源占用</td></tr>
<tr><td style="text-align: left;">资源</td><td style="text-align: left;">强大的计算与存储</td><td style="text-align: left;">有限算力、内存、存储、电池</td></tr>
</table>

**表格解读**：
- **环境维度**：训练通常在云端GPU/NPU服务器上完成，拥有充足电力和散热条件；而推理往往部署在终端设备（手机、摄像头、车载芯片）上，受体积、功耗、散热严格约束。
- **关注点维度**：训练阶段以提升模型精度为首要目标，可以反复迭代；推理阶段则更关注实时性（延迟低）、吞吐量（每秒处理请求数）、功耗（电池续航）和内存占用（不OOM）。
- **资源维度**：训练服务器可配备数百GB显存和TB级存储；边缘设备可能只有几MB可用内存，这直接决定了模型必须经过压缩优化才能部署。

> **核心技术难题**：如何在保持较高精度的前提下，显著降低计算开销、内存占用与能耗。这一问题贯穿整个部署流程，后续的量化、剪枝、算子融合等优化技术都是为解决此难题而设计。

### 1.2 智能系统部署的两类技术路线

当前深度学习模型部署主要有两条技术路线，两者协同工作：

**路线一：面向深度学习的智能编译器方案**
- 核心思想：构建专为深度学习设计的编译器，自动解析模型结构与计算特征，结合目标硬件实现算子融合、内存优化、调度策略等自动化优化
- 代表：Apache TVM、华为昇腾 CANN 编译器、Google XLA
- 优势：开发者无须手工适配底层硬件，跨平台性强

**路线二：推理框架与硬件加速库方案**
- 核心思想：依托为特定硬件深度优化的推理框架和基础算子库，通过调用优化后的库函数快速实现部署
- 代表：TensorRT（NVIDIA）、MindIE（昇腾）、ONNX Runtime、OpenVINO（Intel）
- 优势：高度优化的硬件加速库，丰富的算子支持

> 两者协同：智能编译器负责自动代码生成与优化，硬件加速库提供高性能基础算子，共同促进从"云端训练"到"边缘部署"的落地。

### 1.3 常见推理框架一览

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">框架</th>
<th style="text-align: left;">厂商/来源</th>
<th style="text-align: left;">特点</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">TensorFlow Lite</td>
<td style="text-align: left;">Google</td>
<td style="text-align: left;">跨平台、灵活量化</td>
<td style="text-align: left;">移动应用、嵌入式</td>
</tr>
<tr>
<td style="text-align: left;">NCNN</td>
<td style="text-align: left;">腾讯</td>
<td style="text-align: left;">无第三方依赖、移动端优化</td>
<td style="text-align: left;">移动端图像处理</td>
</tr>
<tr>
<td style="text-align: left;">MNN</td>
<td style="text-align: left;">阿里巴巴</td>
<td style="text-align: left;">多硬件支持、轻量级</td>
<td style="text-align: left;">跨平台移动应用</td>
</tr>
<tr>
<td style="text-align: left;">TensorRT</td>
<td style="text-align: left;">NVIDIA</td>
<td style="text-align: left;">算子融合、极致GPU性能</td>
<td style="text-align: left;">NVIDIA GPU部署</td>
</tr>
<tr>
<td style="text-align: left;">MindIE</td>
<td style="text-align: left;">华为昇腾</td>
<td style="text-align: left;">与CANN深度集成</td>
<td style="text-align: left;">昇腾处理器推理</td>
</tr>
<tr>
<td style="text-align: left;">ONNX Runtime</td>
<td style="text-align: left;">Microsoft</td>
<td style="text-align: left;">跨平台、多执行提供程序</td>
<td style="text-align: left;">跨平台模型部署</td>
</tr>
<tr>
<td style="text-align: left;">OpenVINO</td>
<td style="text-align: left;">Intel</td>
<td style="text-align: left;">英特尔硬件深度优化</td>
<td style="text-align: left;">工业视觉、边缘计算</td>
</tr>
</table>

**表格解读**：
- **TensorFlow Lite / NCNN / MNN**：三者均面向移动端和嵌入式设备，共同特点是轻量级、依赖少。NCNN由腾讯开源，在移动端图像处理（如人脸检测）中应用广泛；MNN由阿里巴巴开源，支持多硬件后端，适合需要跨Android/iOS部署的场景。
- **TensorRT**：NVIDIA官方推理引擎，针对GPU做了深度优化（如算子融合、精度校准），在NVIDIA GPU上能获得最佳性能，但绑定NVIDIA硬件。
- **MindIE**：华为昇腾专用推理引擎，与CANN算子库深度集成，在昇腾NPU上可获得最优性能，是本课程关注的重点。
- **ONNX Runtime**：微软开源的跨平台推理框架，支持CPU/GPU/NPU等多种执行后端，灵活性最高但针对性优化不如厂商专用框架。
- **OpenVINO**：Intel面向工业视觉和边缘计算的推理框架，对Intel CPU/iGPU/VPU做了深度优化。

> **选型建议**：根据硬件平台选择深度优化的框架（昇腾选MindIE/CANN，NVIDIA选TensorRT，Intel选OpenVINO），跨平台灵活性选ONNX Runtime。选型核心原则是"硬件与框架匹配"——专用框架在对应硬件上性能最优，跨平台框架灵活性最好但性能折中。

---

## 2. 智能系统部署流程

深度学习模型从训练到部署的完整流程分为四个阶段：

```
PyTorch模型(.pth) → ONNX中间格式(.onnx) → ATC编译 → 昇腾OM模型(.om) → ACL推理
   训练框架         跨框架交换格式       模型编译优化     NPU离线推理    端侧执行
```

### 四个关键阶段

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">内容</th>
<th style="text-align: left;">关键操作</th>
</tr>
<tr>
<td style="text-align: left;">1. 模型准备</td>
<td style="text-align: left;">导出为ONNX格式</td>
<td style="text-align: left;">算子融合、常量折叠、冗余裁剪、量化优化</td>
</tr>
<tr>
<td style="text-align: left;">2. 平台适配</td>
<td style="text-align: left;">转换为目标平台格式</td>
<td style="text-align: left;">昇腾用ATC转OM，NVIDIA用TensorRT转Engine</td>
</tr>
<tr>
<td style="text-align: left;">3. 推理执行</td>
<td style="text-align: left;">加载模型、预处理、执行</td>
<td style="text-align: left;">尺寸调整、归一化、格式转换</td>
</tr>
<tr>
<td style="text-align: left;">4. 结果处理</td>
<td style="text-align: left;">后处理与系统集成</td>
<td style="text-align: left;">解码、过滤、业务集成</td>
</tr>
</table>

**四阶段详解**：
- **阶段1（模型准备）**：训练得到的PyTorch模型（.pth）包含权重和计算图，但依赖PyTorch运行时。导出为ONNX格式后，模型变为独立的标准格式，不再依赖任何训练框架。此阶段可做常量折叠（将编译期可确定的计算提前求值）、冗余节点裁剪等图优化。
- **阶段2（平台适配）**：ONNX是通用中间格式，但不同硬件有各自的最优执行格式。昇腾平台使用ATC工具将ONNX编译为OM离线模型，编译过程中完成算子映射、图融合、内存规划等针对硬件的深度优化；NVIDIA平台则用TensorRT将ONNX转为Engine文件。
- **阶段3（推理执行）**：加载编译后的模型，对输入数据做预处理（缩放、归一化、格式转换），然后送入模型执行前向推理。此阶段的关键是预处理要与训练时保持一致，否则精度会下降。
- **阶段4（结果处理）**：模型输出通常是原始logits或概率向量，需要根据任务做后处理（如分类任务取argmax、检测任务做NMS），最后将结果集成到业务系统中。

### 2.1 动手实践：检查昇腾部署环境

在开始部署之前，先检查当前昇腾 NPU 环境是否就绪。以下代码在 notebook 中可以直接运行：

**代码功能说明**：
1. **NPU硬件检查**：调用 `npu-smi info` 命令查看NPU设备状态，类似GPU环境下的 `nvidia-smi`。若返回正常则说明NPU驱动已安装且设备可用。
2. **ATC工具检查**：ATC是CANN提供的模型编译工具，用于将ONNX/Caffe等模型转为昇腾OM格式。检查ATC是否可用决定了后续能否做模型转换。
3. **环境变量检查**：`ASCEND_HOME_PATH` 是CANN工具链的安装路径，未设置则说明CANN环境变量未加载（需执行 `source set_env.sh`）。
4. **Python依赖检查**：检查 `torch`、`numpy`、`onnx` 三个核心库是否安装。其中 `onnx` 库是导出ONNX模型的必需依赖。

**预期结果**：在昇腾NPU环境中，NPU硬件和ATC工具应检测成功，三个Python库均显示版本号。若某项显示 `[SKIP]`，说明对应组件未安装或未配置，需根据提示安装。

In [ ]:
!pip install "numpy<2.0" -q
import os, sys, subprocess

print('=' * 60)
print('昇腾部署环境检查')
print('=' * 60)

# 1. 检查 NPU 硬件
try:
    result = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print('[OK] NPU 硬件检测成功')
        for line in result.stdout.strip().split('\n')[:10]:
            print(f'  {line}')
    else:
        print('[SKIP] npu-smi 不可用')
except Exception as e:
    print(f'[SKIP] npu-smi 未安装: {e}')

# 2. 检查 ATC 工具
try:
    result = subprocess.run(['atc'], capture_output=True, text=True, timeout=10)
    if 'framework' in result.stderr or 'framework' in result.stdout:
        print('\n[OK] ATC 工具可用')
    else:
        print('\n[SKIP] ATC 工具未配置')
except Exception as e:
    print(f'\n[SKIP] ATC 未安装: {e}')

# 3. 检查 CANN 环境变量
ascend_home = os.environ.get('ASCEND_HOME_PATH', '')
print(f'\nASCEND_HOME_PATH: {ascend_home or "(未设置)"}')

# 4. 检查 Python 依赖
for pkg in ['torch', 'numpy', 'onnx']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', '?')
        print(f'[OK] {pkg} {ver}')
    except ImportError:
        print(f'[SKIP] {pkg} 未安装')

print('=' * 60)

---

## 3. CANN 异构计算架构

<img src="../../images/cann_software_architecture.png" alt="CANN架构" style="display: block; margin-left: 0;" />

### 3.1 什么是 CANN

CANN（Compute Architecture for Neural Networks）是华为针对 AI 场景推出的异构计算架构，对上支持多种 AI 框架，对下服务 AI 处理器与编程，发挥承上启下的关键作用。

- **对上**：无缝对接 MindSpore、PyTorch、TensorFlow 等主流框架
- **对下**：深度理解昇腾 AI 处理器设计，将 AI 运算转化为最适合硬件执行的指令流
- **全栈协同**：集成 HCCL 通信库、DVPP 视觉预处理、AOE 调优工具

### 3.2 CANN 三层逻辑架构

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层</th>
<th style="text-align: left;">名称</th>
<th style="text-align: left;">核心功能</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">应用层</td>
<td style="text-align: left;">开发者入口，对接 AI 框架与工具链（AscendCL、MindStudio）</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">芯片使能层</td>
<td style="text-align: left;">核心能力引擎：图引擎GE、算子库、运行时Runtime、HCCL、DVPP</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">计算资源层</td>
<td style="text-align: left;">昇腾 AI 处理器（AICore/AICPU/DVPP）及驱动程序</td>
</tr>
</table>

**三层架构详解**：
- **应用层（第1层）**：这是开发者直接接触的层面。通过AscendCL（C语言接口）或MindStudio（IDE工具），开发者可以加载模型、执行推理、调用算子。PyTorch、MindSpore等框架也通过此层接入CANN。
- **芯片使能层（第2层）**：CANN的核心能力所在。图引擎GE负责计算图的编译优化（算子融合、内存复用、流水并行）；算子库提供1400+高性能算子实现；Runtime管理设备资源和任务调度；HCCL提供多卡通信能力；DVPP负责图像/视频硬件加速预处理。
- **计算资源层（第3层）**：最底层的硬件资源。AICore是AI计算核心（包含Cube矩阵计算单元和Vector向量计算单元），AICPU负责非AI计算任务，DVPP是数字视觉预处理模块。驱动程序管理这些硬件资源的调度和通信。

> CANN 上接 AI 框架与开发工具，下驱昇腾 AI 处理器，是全栈的核心枢纽。三层架构的设计使得框架适配（上层）与硬件优化（下层）解耦，开发者只需关注应用层接口，底层的硬件细节由CANN自动处理。

### 3.3 CANN vs CUDA

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">维度</th>
<th style="text-align: left;">CANN</th>
<th style="text-align: left;">CUDA</th>
</tr>
<tr>
<td style="text-align: left;">硬件适配</td>
<td style="text-align: left;">华为昇腾 AI 芯片</td>
<td style="text-align: left;">NVIDIA GPU</td>
</tr>
<tr>
<td style="text-align: left;">生态成熟度</td>
<td style="text-align: left;">快速发展中</td>
<td style="text-align: left;">成熟</td>
</tr>
<tr>
<td style="text-align: left;">优化深度</td>
<td style="text-align: left;">针对昇腾硬件深度优化</td>
<td style="text-align: left;">针对 NVIDIA 优化</td>
</tr>
<tr>
<td style="text-align: left;">编程语言</td>
<td style="text-align: left;">AscendC (C++扩展)</td>
<td style="text-align: left;">CUDA C/C++</td>
</tr>
<tr>
<td style="text-align: left;">编译器</td>
<td style="text-align: left;">毕昇编译器</td>
<td style="text-align: left;">NVCC</td>
</tr>
</table>

**对比解读**：CUDA是NVIDIA历经20年打造的GPU计算生态，工具链成熟、社区资源丰富。CANN是华为针对昇腾AI芯片打造的异构计算架构，虽然生态成熟度尚在追赶中，但针对昇腾硬件做了深度协同优化——从算子库到编译器到驱动全栈自研，能充分发挥昇腾芯片的AI计算能力。两者在编程模型上高度相似（都是Host+Device异构模型），使得CUDA开发者迁移到CANN的学习成本较低。

### 3.4 CANN 开源生态

2025年8月，华为正式宣布 CANN 全面开源开放，标志着国产 AI 计算架构自主化进入新阶段：
- 1400+ 高性能算子
- 900+ 预训练模型
- CUDA 转译层（降低迁移成本）
- 开源地址：https://atomgit.com/cann

**开源意义**：CANN开源意味着开发者可以查看算子实现源码、参与社区贡献、基于源码做定制化优化。CUDA转译层大幅降低了从NVIDIA生态迁移到昇腾生态的成本——原有CUDA代码可以较容易地转译为CANN/AscendC代码，加速了国产AI计算生态的发展。

### 3.5 动手实践：PyTorch 与昇腾 NPU 的零门槛迁移

CANN 通过 `torch_npu` 适配层，让 PyTorch 代码只需一行 `.npu()` 即可迁移到昇腾 NPU：

**代码功能说明**：
1. 导入 `torch_npu` 后，PyTorch即可识别NPU设备。`torch.npu.is_available()` 检查NPU是否可用，类似 `torch.cuda.is_available()`。
2. 通过 `.npu()` 将张量从CPU迁移到NPU设备，语法与 `.cuda()` 完全一致，体现了torch_npu的设计理念——最小化迁移成本。
3. 代码在NPU和CPU上分别执行1000×1000矩阵乘法，对比耗时。`torch.npu.synchronize()` 用于等待NPU异步计算完成（类似 `torch.cuda.synchronize()`），确保计时准确。

**预期结果**：NPU执行矩阵乘法的速度应显著快于CPU（通常快数十倍），加速比取决于矩阵规模和NPU型号。对于小规模矩阵，由于数据搬运开销占比大，加速比可能不明显；规模越大，NPU的并行计算优势越突出。

**结果解释**：NPU的AICore包含专用的Cube矩阵计算单元，矩阵乘法被映射到硬件矩阵乘加电路上直接执行，远快于CPU的标量/向量计算。这就是为什么深度学习推理（本质是大量矩阵运算）在NPU上能获得巨大加速。

In [ ]:
import torch
import torch_npu

print(f'PyTorch: {torch.__version__}')
print(f'NPU 可用: {torch.npu.is_available()}')
if torch.npu.is_available():
    print(f'NPU 设备: {torch.npu.get_device_name(0)}')

# 在 NPU 上执行矩阵乘法
a = torch.randn(1000, 1000).npu()
b = torch.randn(1000, 1000).npu()
torch.npu.synchronize()

import time
start = time.time()
c = torch.matmul(a, b)
torch.npu.synchronize()
npu_time = time.time() - start

a_cpu = a.cpu(); b_cpu = b.cpu()
start = time.time()
c_cpu = torch.matmul(a_cpu, b_cpu)
cpu_time = time.time() - start

print(f'\n矩阵乘法 (1000x1000):')
print(f'  CPU: {cpu_time*1000:.2f} ms')
print(f'  NPU: {npu_time*1000:.2f} ms')
if npu_time > 0:
    print(f'  加速比: {cpu_time/npu_time:.1f}x')

---

## 4. CANN 核心组件详解

### 4.1 CANN 算子库

CANN 提供 1400+ 高性能算子，按功能分为四大库：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">算子库</th>
<th style="text-align: left;">内容</th>
<th style="text-align: left;">典型算子</th>
</tr>
<tr>
<td style="text-align: left;">数学库 (ops-math)</td>
<td style="text-align: left;">数学类基础算子</td>
<td style="text-align: left;">add, mul, diag, angle</td>
</tr>
<tr>
<td style="text-align: left;">神经网络库 (ops-nn)</td>
<td style="text-align: left;">神经网络高阶算子</td>
<td style="text-align: left;">Conv2d, Pool, MatMul, BatchNorm</td>
</tr>
<tr>
<td style="text-align: left;">计算机视觉库 (ops-cv)</td>
<td style="text-align: left;">图像处理算子</td>
<td style="text-align: left;">grid_sample, resize, crop</td>
</tr>
<tr>
<td style="text-align: left;">Transformer库 (ops-transformer)</td>
<td style="text-align: left;">大模型专用算子</td>
<td style="text-align: left;">flash_attention, MoE, all_gather_matmul</td>
</tr>
</table>

**四大算子库详解**：
- **数学库**：最底层的数学运算，如加减乘除、三角函数、矩阵分解等。这些算子是构建复杂网络的基础，通常映射到Vector计算单元执行。
- **神经网络库**：深度学习核心算子，包括卷积、池化、全连接、归一化等。卷积算子映射到Cube单元（矩阵乘加专用电路），是CNN推理性能的关键。
- **计算机视觉库**：图像预处理算子，如缩放、裁剪、仿射变换、光流估计等。这些算子常用于推理前的数据预处理，部分可由DVPP硬件加速。
- **Transformer库**：针对大语言模型（LLM）优化的专用算子，如FlashAttention（注意力加速）、MoE（混合专家）、all_gather_matmul（通信+计算融合）。这些算子是CANN支持大模型推理的关键。

算子调用方式：`aclnn` API（以 aclnn 为前缀的 C 语言接口），无须提供 IR 定义，方便高效构建模型。开发者既可以通过框架（PyTorch/MindSpore）间接调用这些算子，也可以通过AscendCL直接调用单个算子实现自定义推理逻辑。

### 4.2 通信库 HCCL

<img src="../../images/hccl.png" alt="HCCL" style="display: block; margin-left: 0;" />

HCCL（Huawei Collective Communication Library）是基于昇腾硬件设计的高性能集合通信库：
- **通信原语**：AllReduce、Broadcast、Allgather、ReduceScatter、AlltoAll
- **通信算法**：Ring（环形）、Mesh（网格）、HD（层次化）
- **高速链路**：HCCS、RoCE、PCIe
- **编程接口**：C 和 Python 两种语言接口

### 4.3 图引擎 GE

<img src="../../images/ge_graph_engine.jpg" alt="图引擎" style="display: block; margin-left: 0;" />

图引擎（Graph Engine）三大关键特性：
1. **统一图开发接口**：支持多框架开发及迁移
2. **图编译优化**：自动流水并行、多算子自动融合、内存优化
3. **图加载执行**：计算图执行下沉到 NPU 侧，减少主机干预

图优化技术包括：常量折叠、公共子表达式消除、剪枝、死边消除、多流并行、内存复用。

### 4.4 昇腾编程语言 AscendC

AscendC 是面向昇腾 AI 处理器算子开发的专用编程语言，基于 C++17 标准扩展的 DSL：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">特性</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">原生 C/C++ 支持</td>
<td style="text-align: left;">完全兼容 C/C++ 标准规范</td>
</tr>
<tr>
<td style="text-align: left;">多层级 API</td>
<td style="text-align: left;">基础API、高阶API、模板库</td>
</tr>
<tr>
<td style="text-align: left;">自动并行</td>
<td style="text-align: left;">SPMD 并行编程模型</td>
</tr>
<tr>
<td style="text-align: left;">孪生调试</td>
<td style="text-align: left;">CPU 模拟调试功能</td>
</tr>
<tr>
<td style="text-align: left;">硬件亲和</td>
<td style="text-align: left;">深度融合 Cube/Vector 单元</td>
</tr>
</table>

**特性详解**：
- **多层级API**：基础API提供对硬件的精细控制（如直接操作Cube/Vector单元），高阶API封装常用计算模式（如MatMul、Softmax），模板库提供泛化算子模板。开发者可根据需要选择不同抽象层级。
- **自动并行（SPMD）**：单程序多数据模型，开发者写一份代码，编译器自动将其并行化到多个AI Core上执行，类似CUDA的线程模型。
- **孪生调试**：在CPU上模拟NPU执行行为，便于在没有NPU硬件时调试算子代码，大幅提升开发效率。

关键语法：`__global__`（设备端内核函数）、`__aicore__`（AICore 限定符）

### 4.5 毕昇编译器

<img src="../../images/bisheng_compiler.png" alt="毕昇编译器" style="display: block; margin-left: 0;" />

毕昇编译器专为昇腾 AI 处理器设计，基于 LLVM 架构：
- 异构编译架构（Host+Device）
- 软硬协同微架构精准优化
- 微指令亲和调度
- 编译命令示例：`bisheng -O2 --cce-soc-version=Ascend910B ...`

### 4.6 运行时与 AscendCL

<img src="../../images/acl_runtime.jpg" alt="ACL运行时" style="display: block; margin-left: 0;" />

AscendCL（Ascend Computing Language）是昇腾计算开放编程框架，提供：
- **运行时管理**：设备初始化、内存分配、Stream 管理
- **模型管理**：OM 模型加载、推理执行
- **单算子调用**：直接调用优化后的算子
- **媒体数据处理**：图像/视频预处理（DVPP）

三大应用场景：终端应用开发、AI 框架集成、高阶功能库封装

### 4.7 驱动程序

昇腾 NPU 驱动三大核心功能：
1. **AI 计算资源统一管理**：统一内存管理（HBM/DDR）、硬件调度器管理
2. **高性能通信**：HDC（Host-Device）、P2P（Device-Device）通信
3. **设备管理与运维**：npu-smi 管理工具、标准化北向接口

---

## 5. CANN 应用案例：模型部署全流程

### 5.1 模型部署链路

```
PyTorch (.pth) → ONNX (.onnx) → ATC → OM (.om) → AscendCL 推理
```

下面用一段完整的代码演示从 PyTorch 模型到 ONNX 导出的过程：

**部署链路各环节作用**：
- **PyTorch (.pth)**：训练产出的模型权重文件，依赖PyTorch框架才能运行，无法直接在NPU上高效执行。
- **ONNX (.onnx)**：开放神经网络交换格式，将模型表示为标准算子图，解耦训练框架与推理硬件。这是"中间人"角色——训练框架只需导出ONNX，推理引擎只需支持ONNX输入。
- **ATC编译**：昇腾模型编译工具，将ONNX编译为OM格式。编译过程中做算子映射（将标准算子映射到昇腾硬件实现）、图融合（合并相邻算子减少内存搬运）、内存优化等。
- **OM (.om)**：昇腾离线模型格式，包含已编译的计算图和权重，可直接在NPU上加载执行，无需运行时编译。
- **AscendCL推理**：通过ACL接口加载OM模型、管理内存、执行推理，是端侧部署的最终执行方式。

In [ ]:
# 定义一个简单的 CNN 模型（MNIST 手写数字识别）
import torch
import torch.nn as nn
import os

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleCNN()
model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f'模型参数量: {total_params:,}')
print(f'模型大小(FP32): {total_params * 4 / 1024:.1f} KB')

In [ ]:
# 导出 ONNX 模型
# 注意：torch.onnx.export() 内部依赖 onnx 库来序列化模型，
# 若 onnx 未安装则导出会抛出 OnnxExporterError，因此先做依赖检查。
dummy_input = torch.randn(1, 1, 28, 28)
onnx_path = 'models/simplecnn.onnx'
os.makedirs('models', exist_ok=True)

try:
    import onnx  # torch.onnx.export 内部需要此库
except ImportError:
    onnx = None
    print('[SKIP] onnx 库未安装，无法导出 ONNX 模型')
    print('请先执行安装：pip install onnx')
    print('安装后重新运行本单元格即可完成导出')

if onnx is not None:
    torch.onnx.export(
        model, dummy_input, onnx_path,
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
        opset_version=11, dynamo=False
    )
    print(f'ONNX 已导出: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

    # 验证 ONNX 模型结构与算子是否合法
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print('ONNX 格式验证通过')

### 5.2 ATC 模型转换

将 ONNX 模型转换为昇腾 OM 离线模型，使用 ATC 工具：

```bash
atc --framework=5 \
    --model=models/simplecnn.onnx \
    --output=models/simplecnn \
    --soc_version=Ascend910B3 \
    --input_shape="input:1,1,28,28" \
    --log=error
```

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;">--framework=5</td>
<td style="text-align: left;">输入为 ONNX 格式</td>
</tr>
<tr>
<td style="text-align: left;">--soc_version</td>
<td style="text-align: left;">目标芯片型号（须与硬件一致）</td>
</tr>
<tr>
<td style="text-align: left;">--input_shape</td>
<td style="text-align: left;">输入节点名:形状（须与 ONNX 导出一致）</td>
</tr>
</table>

**参数详解**：
- **--framework=5**：指定输入模型格式。5代表ONNX，其他如3代表TensorFlow，0代表Caffe。
- **--soc_version**：目标芯片型号，必须与实际硬件一致。型号错误会导致编译失败或生成的OM模型无法运行。可通过 `npu-smi info` 查询实际芯片型号。
- **--input_shape**：指定输入节点的名称和形状。节点名必须与ONNX导出时的 `input_names` 完全一致（本例为 `input`），形状为 `1,1,28,28`（batch=1, channel=1, height=28, width=28，对应MNIST单通道28×28图像）。
- **--log=error**：只输出错误级别日志，减少日志干扰。调试时可改为 `--log=info` 或 `--log=debug` 获取更详细的信息。

> 看到 `ATC run success` 即转换成功，此时 `models/simplecnn.om` 文件已生成。若报错，常见原因包括：ONNX文件路径错误、soc_version与硬件不匹配、input_shape节点名不一致、ONNX包含不支持的算子等。

In [ ]:
# 在 notebook 中执行 ATC 转换
import subprocess

onnx_path = 'models/simplecnn.onnx'
soc_version = 'Ascend910B3'  # 与硬件平台一致

# 先检查 ONNX 模型是否存在，避免 ATC 报 realpath error
if not os.path.exists(onnx_path):
    print(f'[SKIP] ONNX 模型不存在: {onnx_path}')
    print('请先成功运行上方"导出 ONNX 模型"单元格（需安装 onnx 库：pip install onnx）')
else:
    cmd = ('atc --framework=5 --model=' + onnx_path + ' '
           '--output=models/simplecnn --soc_version=' + soc_version + ' '
           '--input_shape="input:1,1,28,28" --log=error')
    print(f'执行命令:\n{cmd}\n')

    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        if result.stderr:
            print(f'[STDERR]: {result.stderr[-300:]}')
        om_path = 'models/simplecnn.om'
        if os.path.exists(om_path):
            print(f'\n[OK] OM 模型已生成: {om_path} ({os.path.getsize(om_path)/1024:.1f} KB)')
        else:
            print('\n[INFO] OM 未生成（可能需要在昇腾环境中运行）')
    except Exception as e:
        print(f'[SKIP] {e}')
        print('请在昇腾 NPU 环境中运行此单元格')

---

## 6. 深度学习模型优化技术

部署前对模型做优化，目的是在不明显损失精度的前提下减小体积、提升速度：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">原理</th>
<th style="text-align: left;">效果</th>
</tr>
<tr>
<td style="text-align: left;"><strong>量化</strong></td>
<td style="text-align: left;">降低参数数值精度（FP32→INT8）</td>
<td style="text-align: left;">体积减75%，速度提升2-4倍</td>
</tr>
<tr>
<td style="text-align: left;"><strong>剪枝</strong></td>
<td style="text-align: left;">移除不重要的参数/结构</td>
<td style="text-align: left;">体积减30-70%，速度提升1.5-3倍</td>
</tr>
<tr>
<td style="text-align: left;"><strong>算子融合</strong></td>
<td style="text-align: left;">合并连续算子为单算子</td>
<td style="text-align: left;">内存访问减60-80%，延迟降30-50%</td>
</tr>
<tr>
<td style="text-align: left;"><strong>知识蒸馏</strong></td>
<td style="text-align: left;">小模型模仿大模型</td>
<td style="text-align: left;">体积减50-90%，速度提升2-5倍</td>
</tr>
</table>

**四种优化技术详解**：
- **量化**：将模型参数从32位浮点（FP32）降低到8位整数（INT8）甚至4位。由于INT8数据宽度仅为FP32的1/4，模型体积直接减少75%，且整数运算在硬件上执行更快。代价是可能引入精度损失，通常需要校准数据集来确定最佳量化参数。昇腾平台使用AMCT工具做量化。
- **剪枝**：分析权重重要性，将接近零的权重置零（非结构化剪枝）或直接移除整个通道/层（结构化剪枝）。非结构化剪枝产生稀疏矩阵，需要硬件/框架支持稀疏计算才能加速；结构化剪枝直接减少计算量，加速效果更实际。
- **算子融合**：将连续的多个小算子合并为一个大算子。例如将Conv+BN+ReLU三个算子融合为一个算子执行，消除了中间结果的内存写入和读取，大幅减少内存带宽压力。ATC编译过程中会自动做算子融合。
- **知识蒸馏**：用大模型（教师）指导小模型（学生）训练，让学生模型在保持较小体积的同时逼近教师模型的精度。适用于需要极致压缩但精度要求仍较高的场景。

> 实际应用中常结合使用：先剪枝+量化，再用知识蒸馏微调恢复精度。这种组合策略能在精度损失可控的前提下获得最大的压缩和加速效果。

### 6.1 边缘设备 AI 部署的挑战

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">挑战</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">内存限制</td>
<td style="text-align: left;">微控制器只有几百KB RAM</td>
</tr>
<tr>
<td style="text-align: left;">计算能力</td>
<td style="text-align: left;">MHz级主频，毫瓦级功耗</td>
</tr>
<tr>
<td style="text-align: left;">功耗约束</td>
<td style="text-align: left;">电池供电需极低功耗</td>
</tr>
<tr>
<td style="text-align: left;">模型精度</td>
<td style="text-align: left;">压缩后需保持足够准确率</td>
</tr>
</table>

**边缘部署挑战详解**：
- **内存限制**：物联网微控制器（如Arduino、STM32）的RAM通常只有几十到几百KB，而一个普通CNN模型可能需要数MB内存。这要求必须通过量化（INT8甚至INT4）和剪枝将模型压缩到可用范围内。
- **计算能力**：边缘MCU主频通常在几十到几百MHz，远低于服务器的GHz级CPU和NPU。推理一帧图像可能需要数百毫秒甚至秒级，需要通过轻量化网络设计（如MobileNet、SqueezeNet）来降低计算量。
- **功耗约束**：电池供电设备对功耗极度敏感。NPU/DSP等专用AI加速芯片的能效比（TOPS/W）远高于通用CPU，是边缘AI部署的首选。
- **模型精度**：压缩后的模型精度会下降，需要在"精度-体积-速度"三角中找到平衡点。通常通过量化感知训练（QAT）来恢复精度。

应对策略：模型压缩与量化、轻量化网络设计（MobileNet）、硬件加速（NPU/DSP）、算子优化、内存管理、异构计算。实际部署中需根据具体设备的资源约束，选择合适的优化组合策略。

---

## 7. 动手实践：AscendCL 推理验证

使用 AscendCL Python 接口加载 OM 模型并执行推理：

**AscendCL推理流程说明**：

AscendCL推理遵循"初始化→加载模型→准备数据→执行推理→释放资源"的标准流程：
1. **初始化**：`acl.init()` 初始化ACL运行时，`acl.rt.set_device(0)` 指定使用0号NPU设备，`acl.rt.create_context()` 创建执行上下文。
2. **加载模型**：读取OM文件到内存，通过 `acl.mdl.load_from_mem()` 加载为模型ID。之后查询模型描述获取输入/输出大小。
3. **准备数据**：在Device侧分配输入/输出内存（`acl.rt.malloc`），创建Dataset对象封装内存缓冲区。将Host侧的输入数据通过 `acl.rt.memcpy` 拷贝到Device侧。
4. **执行推理**：`acl.mdl.execute()` 提交推理任务，`acl.rt.synchronize_stream()` 等待执行完成。推理结果在Device侧输出内存中，需拷贝回Host侧才能读取。
5. **释放资源**：按"后创建先释放"顺序销毁所有资源（stream、dataset、内存、模型、上下文），最后调用 `acl.finalize()`。

**预期结果**：在昇腾CANN环境中，代码会输出预测的数字类别（0-9）和对应的logits向量。若OM模型不存在或acl未安装，会输出相应的SKIP提示。由于此处使用随机输入（未训练的模型），预测结果无实际意义，仅验证推理流程是否畅通。

In [ ]:
# AscendCL 推理流程演示
ACL_SUCCESS = 0
ACL_MEM_MALLOC_NORMAL_ONLY = 2
ACL_MEMCPY_HOST_TO_DEVICE = 1
ACL_MEMCPY_DEVICE_TO_HOST = 2

HAS_ACL = False
try:
    import acl
    HAS_ACL = True
    print('[OK] AscendCL (acl) 已加载')
except ImportError:
    print('[SKIP] acl 未安装（需要在昇腾 CANN 环境中运行）')

if HAS_ACL:
    import numpy as np
    om_path = 'models/simplecnn.om'
    if os.path.exists(om_path):
        acl.init()
        acl.rt.set_device(0)
        context, _ = acl.rt.create_context(0)
        with open(om_path, 'rb') as f:
            om_bytes = f.read()
        ptr = acl.util.bytes_to_ptr(om_bytes)
        model_id, _ = acl.mdl.load_from_mem(ptr, len(om_bytes))
        print(f'[OK] OM 模型已加载: {om_path}')
        model_desc = acl.mdl.create_desc()
        acl.mdl.get_desc(model_desc, model_id)
        input_size = acl.mdl.get_input_size_by_index(model_desc, 0)
        output_size = acl.mdl.get_output_size_by_index(model_desc, 0)
        if output_size == 0:
            output_size = 10 * 4
        print(f'  input_size={input_size}, output_size={output_size}')
        in_buf, _ = acl.rt.malloc(input_size, ACL_MEM_MALLOC_NORMAL_ONLY)
        out_buf, _ = acl.rt.malloc(output_size, ACL_MEM_MALLOC_NORMAL_ONLY)
        in_dataset = acl.mdl.create_dataset()
        acl.mdl.add_dataset_buffer(in_dataset, acl.create_data_buffer(in_buf, input_size))
        out_dataset = acl.mdl.create_dataset()
        acl.mdl.add_dataset_buffer(out_dataset, acl.create_data_buffer(out_buf, output_size))
        stream, _ = acl.rt.create_stream()
        test_input = np.random.randn(1, 1, 28, 28).astype(np.float32)
        in_ptr = acl.util.numpy_to_ptr(test_input)
        acl.rt.memcpy(in_buf, input_size, in_ptr, input_size, ACL_MEMCPY_HOST_TO_DEVICE)
        acl.rt.synchronize_stream(stream)
        acl.mdl.execute(model_id, in_dataset, out_dataset)
        acl.rt.synchronize_stream(stream)
        buf = acl.mdl.get_dataset_buffer(out_dataset, 0)
        out_addr = acl.get_data_buffer_addr(buf)
        actual_size = int(acl.get_data_buffer_size(buf)) or output_size
        output_np = np.zeros(actual_size // 4, dtype=np.float32)
        out_ptr = acl.util.numpy_to_ptr(output_np)
        acl.rt.memcpy(out_ptr, actual_size, out_addr, actual_size, ACL_MEMCPY_DEVICE_TO_HOST)
        acl.rt.synchronize_stream(stream)
        pred = int(output_np.argmax())
        print(f'\n推理结果: 预测数字 = {pred}')
        print(f'输出 logits: {output_np[:10]}')
        acl.rt.destroy_stream(stream)
        acl.mdl.destroy_dataset(in_dataset); acl.mdl.destroy_dataset(out_dataset)
        acl.rt.free(in_buf); acl.rt.free(out_buf)
        acl.mdl.destroy_desc(model_desc); acl.mdl.unload(model_id)
        acl.rt.destroy_context(context); acl.rt.reset_device(0); acl.finalize()
        print('\n[OK] 资源已释放，推理完成')
    else:
        print(f'[SKIP] OM 模型不存在: {om_path}')
        print('请先运行上方的 ATC 转换单元格')
else:
    print('\n请在昇腾 CANN 环境中运行此单元格以体验 AscendCL 推理')

---

## 8. 总结

<table style="text-align: left; margin-left: 0;">
<tr><th style="text-align: left;">概念</th><th style="text-align: left;">一句话理解</th></tr>
<tr><td style="text-align: left;">智能系统部署</td><td style="text-align: left;">将训练好的模型从训练环境迁移到推理硬件的全过程</td></tr>
<tr><td style="text-align: left;">两类技术路线</td><td style="text-align: left;">智能编译器（自动优化）+ 推理框架/加速库（高性能算子）</td></tr>
<tr><td style="text-align: left;">部署链路</td><td style="text-align: left;">.pth → .onnx → ATC → .om → ACL 推理</td></tr>
<tr><td style="text-align: left;">CANN</td><td style="text-align: left;">华为昇腾异构计算架构，承上启下的核心桥梁</td></tr>
<tr><td style="text-align: left;">AscendC</td><td style="text-align: left;">面向昇腾 NPU 的高性能算子编程语言</td></tr>
<tr><td style="text-align: left;">毕昇编译器</td><td style="text-align: left;">专为昇腾处理器设计的编译器，基于 LLVM</td></tr>
<tr><td style="text-align: left;">AscendCL</td><td style="text-align: left;">昇腾计算开放编程框架，提供推理/算子/媒体处理接口</td></tr>
<tr><td style="text-align: left;">模型优化</td><td style="text-align: left;">量化+剪枝+融合+蒸馏，减小体积提升速度</td></tr>
</table>

**核心知识点回顾**：
- 智能系统部署的本质是解决"训练环境与推理环境差异"的问题，核心难题是在保持精度的前提下降低计算开销和资源占用。
- 部署链路 `.pth → .onnx → .om` 中，ONNX作为中间格式解耦了训练框架和推理硬件，ATC编译实现了针对昇腾硬件的深度优化。
- CANN全栈（算子库→GE→毕昇编译器→AscendCL→驱动）各组件协同工作，将高层模型描述逐步转化为NPU可执行的底层指令。
- 模型优化技术（量化、剪枝、融合、蒸馏）是部署前的重要环节，实际应用中常组合使用以在精度和性能间取得平衡。

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

**第1题**（单选题）深度学习模型部署的核心技术难题是什么？

- A. 在保持精度的前提下降低计算开销、内存与能耗
- B. 提高模型训练速度
- C. 增加模型参数量
- D. 使用更大的数据集


In [ ]:
q1 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第1题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）智能系统部署的两类技术路线是？

- A. 智能编译器方案 + 推理框架/加速库方案
- B. CPU方案 + GPU方案
- C. 云端方案 + 边缘方案
- D. 训练方案 + 推理方案


In [ ]:
q2 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第2题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）以下哪个是华为昇腾专用的推理框架？

- A. MindIE
- B. TensorRT
- C. OpenVINO
- D. NCNN


In [ ]:
q3 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第3题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）模型部署的标准链路是什么？

- A. .pth → .onnx → ATC → .om → ACL推理
- B. .pth → .tflite → 推理
- C. .onnx → .om → 训练
- D. .pth → .om → 训练


In [ ]:
q4 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第4题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）CANN的三层逻辑架构是？

- A. 应用层 + 芯片使能层 + 计算资源层
- B. 输入层 + 隐藏层 + 输出层
- C. 训练层 + 验证层 + 部署层
- D. 硬件层 + 驱动层 + 应用层


In [ ]:
q5 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第5题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）AscendC编程语言的核心优势是什么？

- A. 原生C/C++支持、多层级API、自动并行
- B. 仅支持Python
- C. 仅支持Java
- D. 不需要编译


In [ ]:
q6 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第6题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）毕昇编译器基于什么架构？

- A. LLVM
- B. GCC
- C. JVM
- D. CLR


In [ ]:
q7 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第7题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）AscendCL提供哪些核心能力？

- A. 运行时管理、模型管理、单算子调用、媒体数据处理
- B. 仅模型训练
- C. 仅数据可视化
- D. 仅网络通信


In [ ]:
q8 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第8题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**第9题**（单选题）FP32转INT8量化的效果是？

- A. 模型体积减少约75%，速度提升2-4倍
- B. 体积不变，速度不变
- C. 体积增加，速度提升
- D. 体积减少，精度完全不变


In [ ]:
q9 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第9题答案已记录：{q9}' if q9 else '⚠️ 请填入答案并运行本单元格')

**第10题**（单选题）边缘设备AI部署的主要挑战不包括？

- A. 拥有充足的计算资源和内存
- B. 内存限制
- C. 计算能力瓶颈
- D. 功耗约束


In [ ]:
q10 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第10题答案已记录：{q10}' if q10 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**

In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / '05_deploy' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_05 import grade
grade(globals())

## 参考资料

- [昇腾 CANN 官方文档](https://www.hiascend.com/document)
- [CANN 开源社区](https://atomgit.com/cann)
- [ATC 工具使用指南](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)
- [AscendCL API 参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition)